# Enhanced Data Cleaning — CNBC, Detik, Kompas

Membersihkan boilerplate/iklan dari kolom `content` untuk 3 sumber berita:

| Source | Pattern yang dihapus |
|--------|----------------------|
| **CNBC**   | Truncate setelah `Selengkapnya saksikan ...` |
| **Detik**  | Truncate setelah `Simak juga ...` / `Saksikan ulasan selengkapnya hanya di ...` / `Simak video ...` / `Scroll to continue` |
| **Kompas** | Hapus inline `Baca juga:...` yang tersebar di tengah artikel |

**Output:** 3 file CSV bersih di `/kaggle/working/` dengan kolom yang sama persis.

## Setup

In [ ]:
import re
import pandas as pd

DATA_PATH   = "/kaggle/input/datasets/davinraffilio9/datalabeled/data_labeled"
OUTPUT_PATH = "/kaggle/working"

print("Libraries loaded.")

## Load Datasets

In [ ]:
df_cnbc   = pd.read_csv(f"{DATA_PATH}/cnbc_labeled.csv")
df_detik  = pd.read_csv(f"{DATA_PATH}/detik_labeled.csv")
df_kompas = pd.read_csv(f"{DATA_PATH}/kompas_labeled.csv")

for name, df in [("CNBC", df_cnbc), ("Detik", df_detik), ("Kompas", df_kompas)]:
    print(f"{name:<8} {df.shape}  |  label dist: {dict(df['label'].value_counts().sort_index())}")

## EDA — Before Cleaning

In [ ]:
# Cek berapa banyak row yang kena masing-masing pattern
checks = {
    "CNBC   — Selengkapnya saksikan"             : (df_cnbc,   r"Selengkapnya saksikan"),
    "Detik  — Simak juga"                        : (df_detik,  r"Simak juga"),
    "Detik  — Saksikan ulasan selengkapnya"      : (df_detik,  r"Saksikan ulasan selengkapnya hanya di"),
    "Detik  — Simak video"                       : (df_detik,  r"[Ss]imak\s+[Vv]ideo"),
    "Detik  — Scroll to continue"                : (df_detik,  r"[Ss]croll\s+to\s+continue"),
    "Kompas — Baca juga"                         : (df_kompas, r"Baca juga"),
}

print(f"{'Pattern':<50} {'Rows':>6}")
print("-" * 60)
for label, (df, pat) in checks.items():
    n = df["content"].str.contains(pat, case=False, na=False, regex=True).sum()
    print(f"{label:<50} {n:>6} / {len(df)}")

print(f"\nAvg content length — CNBC  : {df_cnbc['content'].str.len().mean():.0f} chars")
print(f"Avg content length — Detik : {df_detik['content'].str.len().mean():.0f} chars")
print(f"Avg content length — Kompas: {df_kompas['content'].str.len().mean():.0f} chars")

## Cleaning Functions

In [ ]:
# ── CNBC: potong dari 'Selengkapnya saksikan' ke bawah ──────────────────
CNBC_CUT = [
    r"Selengkapnya saksikan[^\n]*",
]

def clean_cnbc(text: str) -> str:
    if pd.isna(text): return ""
    text = str(text)
    for pat in CNBC_CUT:
        m = re.search(pat, text, flags=re.IGNORECASE | re.DOTALL)
        if m:
            text = text[:m.start()].rstrip()
    return re.sub(r"\s+", " ", text).strip()


# ── Detik: potong dari trigger phrase paling awal ────────────────────────
DETIK_CUT = [
    r"Simak juga[^\n]*",
    r"Saksikan ulasan selengkapnya hanya di[^\n]*",
    r"[Ss]imak\s+[Vv]ideo[^\n]*",
    r"[Ss]croll\s+to\s+continue\s+with\s+content[^\n]*",
]

def clean_detik(text: str) -> str:
    if pd.isna(text): return ""
    text = str(text)
    earliest = len(text)
    for pat in DETIK_CUT:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m and m.start() < earliest:
            earliest = m.start()
    text = text[:earliest].rstrip()
    return re.sub(r"\s+", " ", text).strip()


# ── Kompas: hapus snippet 'Baca juga:...' yang inline ────────────────────
# Lookahead cari transisi ke kalimat baru: [Kapital][4+huruf kecil][spasi][huruf kecil]
BACA_JUGA_RE = re.compile(
    r"(?i)baca\s+juga\s*:[^\n]*?(?=[A-Z][a-z]{3,}\s+[a-z]|\n|$)"
)

def clean_kompas(text: str) -> str:
    if pd.isna(text): return ""
    text = BACA_JUGA_RE.sub("", str(text))
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


print("Cleaning functions ready.")

## Apply Cleaning

In [ ]:
df_cnbc["content"]   = df_cnbc["content"].apply(clean_cnbc)
df_detik["content"]  = df_detik["content"].apply(clean_detik)
df_kompas["content"] = df_kompas["content"].apply(clean_kompas)

# Rebuild kolom text = title + content
for df in [df_cnbc, df_detik, df_kompas]:
    df["text"] = (
        df["title"].astype(str).str.strip() + ". " +
        df["content"].astype(str).str.strip()
    ).str.strip()

print("Cleaning done!")

## EDA — After Cleaning

In [ ]:
print(f"{'Source':<8} {'Before':>8} {'After':>8} {'Diff':>8}")
print("-" * 36)

# Reload originals untuk compare panjang
orig_cnbc   = pd.read_csv(f"{DATA_PATH}/cnbc_labeled.csv")
orig_detik  = pd.read_csv(f"{DATA_PATH}/detik_labeled.csv")
orig_kompas = pd.read_csv(f"{DATA_PATH}/kompas_labeled.csv")

for name, orig, cleaned in [
    ("CNBC",   orig_cnbc,   df_cnbc),
    ("Detik",  orig_detik,  df_detik),
    ("Kompas", orig_kompas, df_kompas),
]:
    before = orig["content"].str.len().mean()
    after  = cleaned["content"].str.len().mean()
    print(f"{name:<8} {before:>8.0f} {after:>8.0f} {after-before:>+8.0f}")

# Verifikasi tidak ada sisa pattern
print("\n=== Verifikasi sisa pattern (harus 0 semua) ===")
remaining = [
    ("CNBC   — Selengkapnya saksikan"       , df_cnbc,   r"Selengkapnya saksikan"),
    ("Detik  — Simak juga"                  , df_detik,  r"Simak juga"),
    ("Detik  — Saksikan ulasan selengkapnya", df_detik,  r"Saksikan ulasan selengkapnya hanya di"),
    ("Kompas — Baca juga"                   , df_kompas, r"Baca juga"),
]
all_ok = True
for label, df, pat in remaining:
    n = df["content"].str.contains(pat, case=False, na=False).sum()
    status = "OK" if n == 0 else f"MASIH ADA {n} rows!"
    if n > 0: all_ok = False
    print(f"  {label:<45} {status}")
print("\nSemua bersih!" if all_ok else "\nAda yang belum bersih, cek regex.")

## Save Output

In [ ]:
COLS = ["date", "title", "content", "article_id", "text", "label"]

df_cnbc[COLS].to_csv(f"{OUTPUT_PATH}/cnbc_labeled.csv",   index=False, encoding="utf-8")
df_detik[COLS].to_csv(f"{OUTPUT_PATH}/detik_labeled.csv",  index=False, encoding="utf-8")
df_kompas[COLS].to_csv(f"{OUTPUT_PATH}/kompas_labeled.csv", index=False, encoding="utf-8")

print("Saved:")
print(f"  cnbc_labeled.csv   — {len(df_cnbc)} rows")
print(f"  detik_labeled.csv  — {len(df_detik)} rows")
print(f"  kompas_labeled.csv — {len(df_kompas)} rows")
print(f"\nOutput dir: {OUTPUT_PATH}")